In [1]:
import cv2
import numpy as np

In [ ]:
def process_tray(image_path):
    img = cv2.imread(image_path)
    if img is None:
        print(f"Nie można wczytać obrazu: {image_path}")
        return
    
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    
    lower_orange = np.array([0, 100, 100])
    upper_orange = np.array([25, 255, 255])
    
    mask = cv2.inRange(hsv, lower_orange, upper_orange)
    
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    tray_contour = max(contours, key=cv2.contourArea)
    
    output = img.copy()
    cv2.drawContours(output, [tray_contour], -1, (0, 255, 0), 3)

    area = cv2.contourArea(tray_contour)

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.medianBlur(gray, 7)
    
    circles = cv2.HoughCircles(
        blurred, 
        cv2.HOUGH_GRADIENT, 
        dp=1.2, 
        minDist=40, 
        param1=50, 
        param2=35, 
        minRadius=20, 
        maxRadius=60
    )

    coin_count = 0
    if circles is not None:
        circles = np.uint16(np.around(circles))
        coin_count = len(circles[0])
        for i in circles[0, :]:
            cv2.circle(output, (i[0], i[1]), i[2], (255, 0, 0), 3) # okrąg
            cv2.circle(output, (i[0], i[1]), 2, (0, 0, 255), 3)    # środek

    print(f"Obraz: {image_path}")
    print(f"1. Kontur tacki wyznaczony.")
    print(f"2. Pole powierzchni tacki: {area:.2f} px^2")
    print(f"3. Wykryto monet: {coin_count}")
    print("-" * 30)

    cv2.imshow('Detekcja - Projekt 2', output)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
if __name__ == "__main__":
    image_paths = [
        'tray1.jpg',
        'tray2.jpg',
        'tray3.jpg'
    ]
    
    for path in image_paths:
        process_tray(path)